CELL 1 : TẢI THƯ VIỆN

In [ ]:
# Cell 1: Cài đặt các thư viện cốt lõi
# 1. Cài đặt PyTorch - Nền tảng tính toán tensor (cần thiết cho mọi mô hình AI hiện đại)
# --index-url .../cu118: Chỉ định tải phiên bản hỗ trợ CUDA 11.8 để chạy được trên GPU của Colab.
!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118

# 2. Cập nhật thư viện Transformers - "Chợ mô hình" của Hugging Face
# --upgrade: Quan trọng để đảm bảo tải được các mô hình mới như Qwen 2.5.
!pip install -q --upgrade transformers

# 3. Cài đặt các thư viện bổ trợ quan trọng:
# - gradio: Để tạo giao diện web (UI) tương tác.
# - sentence-transformers: Để biến văn bản thành vector (số hóa ý nghĩa).
# - pillow (PIL): Thư viện xử lý hình ảnh cơ bản.
# - accelerate: Giúp tối ưu hóa việc tải và chạy mô hình trên các thiết bị phần cứng (GPU/CPU).
# - bitsandbytes: Kỹ thuật lượng tử hóa (Quantization) 8-bit/4-bit. CỰC KỲ QUAN TRỌNG để chạy mô hình lớn trên GPU có VRAM giới hạn.
!pip install -q gradio sentence-transformers pillow accelerate bitsandbytes

CELL 2 : IMPORT THƯ VIỆN


In [ ]:
# Cell 2: Import các thư viện (Cập nhật cho Qwen và BLIP)
import torch # Thư viện chính để làm việc với Tensor và GPU
import torch.nn.functional as F # Các hàm chức năng (ví dụ: tính xác suất softmax)

# Import từ thư viện Transformers của Hugging Face:
from transformers import (
    AutoTokenizer,        # "Máy dịch" biến văn bản thành số (token) để mô hình hiểu
    AutoModelForCausalLM, # Lớp chung để tải các mô hình ngôn ngữ sinh (như Qwen, GPT)
    AutoProcessor,        # Xử lý dữ liệu đa phương thức (ảnh + text)
    AutoModel,            # Lớp mô hình cơ bản
    pipeline,             # Công cụ "mì ăn liền" cho các tác vụ như phân tích cảm xúc
    BitsAndBytesConfig,   # Cấu hình cho kỹ thuật nén 4-bit (quan trọng!)
    BlipProcessor,        # Xử lý ảnh cho mô hình BLIP
    BlipForConditionalGeneration, # Mô hình BLIP (để sinh mô tả ảnh)
    CLIPProcessor,        # Xử lý ảnh cho mô hình CLIP
    CLIPModel,             # Mô hình CLIP (để hiểu ngữ nghĩa ảnh)
)

# Các thư viện bổ sung:
from sentence_transformers import SentenceTransformer # Tạo vector ngữ nghĩa cho câu
import gradio as gr       # Tạo giao diện web
from PIL import Image     # Xử lý file ảnh
import json               # Làm việc với file JSON (để lưu/đọc lịch sử chat)
import numpy as np        # Tính toán số học
from datetime import datetime # Xử lý thời gian
import os                 # Tương tác với hệ điều hành (kiểm tra file, thư mục)

print("Đã import thư viện thành công!")

Đã import thư viện thành công!


CELL 3 : TẠO QWEN 2.5 VÀ TOKENIZER

In [ ]:
# Cell 3: Khởi tạo mô hình LLM Qwen2.5 (ĐÃ SỬA LỖI PAD TOKEN)
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

print("Đang tải mô hình ngôn ngữ Qwen2.5 (1.5B)...")

model_name = "Qwen/Qwen2.5-1.5B-Instruct"

try:
    # --- CẤU HÌNH NÉN 4-BIT ---
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True,
    )

    # --- TẢI TOKENIZER ---
    tokenizer = AutoTokenizer.from_pretrained(model_name)

    # ==============================================================================
    # --- SỬA LỖI tokenizer ---
    # Kiểm tra nếu tokenizer chưa có pad_token thì gán nó bằng eos_token.
    # Điều này giúp khắc phục cảnh báo "attention mask is not set".
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
        tokenizer.pad_token_id = tokenizer.eos_token_id
    print(f"Đã thiết lập PAD token: {tokenizer.pad_token} (ID: {tokenizer.pad_token_id})")
    # ==============================================================================

    # --- TẢI MÔ HÌNH ---
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        quantization_config=bnb_config,
        device_map="auto",
        trust_remote_code=True
    )
    print("Mô hình Qwen 2.5 đã sẵn sàng chiến đấu!")

except Exception as e:
    print(f"Lỗi tải mô hình 4-bit: {e}")
    print("Đang thử tải chế độ thường (float16)...")
    # Fallback
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    # sửa lỗi pad token ở chế độ fallback này luôn cho chắc
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
        tokenizer.pad_token_id = tokenizer.eos_token_id

    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        torch_dtype=torch.float16,
        device_map="auto"
    )

# Quy trình hoạt động (Khi sử dụng sau này):
# Chiều vào (Encoding): Khi bạn nhập câu "Chào AI Muse", Tokenizer sẽ chặt nhỏ câu này ra: ["Chào", "AI", "Muse"]. Sau đó, nó tra từ điển để biến thành dãy số: [9876, 123, 456]. Đây là ngôn ngữ toán học mà Qwen hiểu.
# Chiều ra (Decoding): Khi Qwen tính toán xong và trả về dãy số [555, 777, 888], Tokenizer lại tra ngược từ điển để dịch thành câu tiếng Việt: "Chào bạn, mình đây!".

Đang tải mô hình ngôn ngữ Qwen2.5 (1.5B)...
Đã thiết lập PAD token: <|endoftext|> (ID: 151643)
Mô hình Qwen 2.5 đã sẵn sàng chiến đấu!


CELL 4 : SENTENCE TRANSFORMERS

In [ ]:
# Cell 4: Mô hình phân tích cảm xúc và embedding
print("Đang tải mô hình phân tích cảm xúc...")

# 1. MÔ HÌNH EMBEDDING (TẠO VECTOR NGỮ NGHĨA) ---
# Tên model: all-MiniLM-L6-v2 (Nhỏ gọn, nhanh, hiệu quả cao)
embedding_model = SentenceTransformer('all-MiniLM-L6-v2')
# Giải thích: Máy tính không hiểu chữ cái như con người. Mô hình này có nhiệm vụ
# biến một câu văn (ví dụ: "Tôi đang buồn") thành một dãy số dài (gọi là vector).
# Các câu có ý nghĩa tương tự nhau sẽ có dãy số tương tự nhau.
# Ứng dụng trong đề tài: Giúp AI Muse so sánh các tác phẩm cũ của bạn, tìm ra các
# ý tưởng lặp lại để đề xuất hướng đi mới ("Phá vỡ lối mòn").

# 2. PIPELINE PHÂN TÍCH CẢM XÚC (SENTIMENT ANALYSIS) ---
# Tên model: twitter-roberta-base-sentiment-latest
# (Được huấn luyện trên hàng triệu dòng tweet nên rất hiểu ngôn ngữ đời thường)
sentiment_analyzer = pipeline(
    "sentiment-analysis", # Tác vụ cần làm là phân tích cảm xúc
    model="cardiffnlp/twitter-roberta-base-sentiment-latest",
    tokenizer="cardiffnlp/twitter-roberta-base-sentiment-latest"
)
# Giải thích: Đây là một "chuyên gia tâm lý" mini. Bạn đưa cho nó một câu,
# nó sẽ phán đoán xem người viết đang có cảm xúc gì (Tích cực, Tiêu cực, hay Trung tính)
# và mức độ tin cậy là bao nhiêu.
# Ứng dụng trong đề tài: Giúp AI Muse biết khi nào bạn đang "bế tắc" hay "buồn"
# để đưa ra lời an ủi hoặc gợi ý phù hợp ("Tạo ra Cảm hứng").

print(" Mô hình phân tích cảm xúc đã sẵn sàng!")

Đang tải mô hình phân tích cảm xúc...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

Device set to use cpu


 Mô hình phân tích cảm xúc đã sẵn sàng!


CELL 5 : BLIP VÀ CLIP

In [ ]:
# Cell 5: Mô hình xử lý hình ảnh (CLIP và BLIP)
print("Đang tải mô hình xử lý hình ảnh (CLIP & BLIP)...")

# Kiểm tra xem có GPU không để chạy cho nhanh
device_str = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Mô hình hình ảnh sẽ chạy trên: {device_str.upper()}")

# (Phần cấu hình bnb_config_image là tùy chọn để nén model nếu bị tràn RAM,
# hiện tại đang được comment lại trong code tải model bên dưới).

# 1. MÔ HÌNH BLIP (IMAGE CAPTIONING - TẠO CHÚ THÍCH ẢNH) ---
# Tên model: Salesforce/blip-image-captioning-base
try:
    print("   ...Đang tải BLIP Image Captioning...")
    # BlipProcessor: Công cụ giúp xử lý ảnh đầu vào cho đúng định dạng của BLIP.
    blip_processor = BlipProcessor.from_pretrained("Salesforce/blip-image-captioning-base")
    # BlipForConditionalGeneration: Mô hình chính. Nhiệm vụ của nó là nhìn ảnh
    # và sinh ra một câu mô tả.
    blip_model = BlipForConditionalGeneration.from_pretrained(
        "Salesforce/blip-image-captioning-base",
        torch_dtype=torch.float16, # Sử dụng độ chính xác 16-bit để tiết kiệm RAM và chạy nhanh trên GPU
        device_map="auto"          # Tự động chia tải
    )
    print(" BLIP Image Captioning đã sẵn sàng!")
except Exception as e:
    print(f" Lỗi tải BLIP: {e}")
    blip_model = None

# 2. MÔ HÌNH CLIP (CONTRASTIVE LANGUAGE-IMAGE PRE-TRAINING) ---
# Tên model: openai/clip-vit-base-patch32
try:
    print("   ...Đang tải CLIP...")
    # CLIPProcessor: Xử lý ảnh và text đầu vào cho CLIP.
    clip_processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")
    # CLIPModel: Mô hình này rất đặc biệt. Nó được học để hiểu mối liên hệ giữa
    # hình ảnh và văn bản. Nó có thể cho biết một bức ảnh khớp với một câu mô tả
    # đến mức nào.
    clip_model = CLIPModel.from_pretrained(
        "openai/clip-vit-base-patch32",
        torch_dtype=torch.float16,
        device_map="auto"
    )
    print("CLIP đã sẵn sàng!")
except Exception as e:
    print(f"Lỗi tải CLIP: {e}")
    clip_model = None

# Ứng dụng trong đề tài:
# - BLIP sẽ "nhìn" bức tranh vẽ hoặc tải lên và mô tả lại nó bằng lời. Sau đó, mô hình Qwen (Cell 3) sẽ dùng lời mô tả đó để đưa ra nhận xét, góp ý về bố cục, màu sắc ("Hoàn thiện Tác phẩm").
# - CLIP (hiện tại chưa dùng nhiều trong code chính nhưng đã tải sẵn) có thể dùng để tìm kiếm các bức tranh có phong cách tương tự, hoặc đánh giá xem bức tranh có phù hợp
#   với một chủ đề (ví dụ: "nỗi buồn") hay không.

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


Đang tải mô hình xử lý hình ảnh (CLIP & BLIP)...
Mô hình hình ảnh sẽ chạy trên: CPU
   ...Đang tải BLIP Image Captioning...


preprocessor_config.json:   0%|          | 0.00/287 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/506 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/389 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/605M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/605M [00:00<?, ?B/s]

CLIP đã sẵn sàng!


CELL 6 : GIỮ ĐOẠN CHAT

In [ ]:
# Cell 6: Hệ thống lưu trữ hồ sơ người dùng
class UserProfile:
    # 1. KHỞI TẠO BỘ NHỚ ---
    def __init__(self):
        # Đây là nơi chứa toàn bộ lịch sử trò chuyện.
        # Nó giống như cuốn nhật ký chung giữa bạn và Nàng thơ.
        self.conversation_history = []

        # Đây là cấu trúc dữ liệu để lưu các đặc điểm cá nhân của người dùng.
        # Hiện tại mới chỉ là khung sườn (placeholder), nhưng trong tương lai,
        # khi AI phân tích lịch sử chat, nó sẽ điền thông tin vào đây.
        # Ví dụ: bạn hay vẽ về "mùa thu" -> lưu vào "favorite_themes".
        self.artistic_preferences = {
            "favorite_themes": [],
            "writing_style": "",
            "emotional_patterns": [],
            "creative_blocks": []
        }
        self.inspiration_sources = []

    # 2. GHI NHỚ TƯƠNG TÁC MỚI ---
    def add_conversation(self, user_input, ai_response, emotion=None):
        # Mỗi khi chat xong một câu, hàm này được gọi để đóng gói
        # mọi thứ (bạn nói gì, AI nói gì, lúc đó bạn vui hay buồn, thời gian nào)
        # vào một "viên gúi ký ức" và cất vào kho conversation_history.
        self.conversation_history.append({
            "timestamp": datetime.now().isoformat(),
            "user": user_input,
            "ai": ai_response,
            "emotion": emotion
        })

    # 3. PHÂN TÍCH THÓI QUEN (Sơ khởi) ---
    def analyze_patterns(self):
        """Phân tích các mẫu sáng tạo từ lịch sử"""
        # Đây là hàm nền móng cho tính năng "Phá vỡ lối mòn".
        # Hiện tại nó chỉ đếm số cuộc trò chuyện gần đây.
        # Nhưng mục tiêu tương lai là: Nó sẽ đọc 10 câu gần nhất của bạn,
        # dùng mô hình Embedding (Cell 4) để xem bạn có đang lặp lại
        # một chủ đề nào đó quá nhiều không để cảnh báo.
        if len(self.conversation_history) < 5:
            return "Chưa đủ dữ liệu để phân tích"

        # Phân tích đơn giản các chủ đề thường xuyên
        recent_texts = [conv["user_input"] for conv in self.conversation_history[-10:]]
        return f"Đã phân tích {len(recent_texts)} cuộc trò chuyện gần đây"

# Khởi tạo hồ sơ người dùng (Tạo ra một bộ nhớ trống sẵn sàng hoạt động)
user_profile = UserProfile()

CELL 7 : TORCH TÍCH HỢP MODULE AI-MUSE CORE

In [ ]:
# Cell 7: Engine xử lý hội thoại chính (Đã tối ưu cho Qwen)
class AIMuseEngine:
    def __init__(self, model, tokenizer, embedding_model, sentiment_analyzer):
        self.model = model
        self.tokenizer = tokenizer
        self.embedding_model = embedding_model
        self.sentiment_analyzer = sentiment_analyzer

    def analyze_emotion(self, text):
        """Phân tích cảm xúc (Cắt ngắn text để xử lý nhanh hơn)"""
        try:
            result = self.sentiment_analyzer(text[:512])[0]
            return result['label'], result['score']
        except:
            return "NEUTRAL", 0.5

    def generate_response(self, user_input, history=None):
        """Tạo phản hồi sử dụng Chat Template chuẩn của Qwen"""

        # 1. Phân tích cảm xúc
        emotion, confidence = self.analyze_emotion(user_input)

        # 2. Tạo System Prompt (Chỉ dẫn nhập vai) - ĐÃ SỬA ĐỔI
        # Định hình vai diễn Nàng thơ với cách xưng hô Nàng - Anh
        system_prompt = (
            "Bạn là AI Muse - một nàng thơ nghệ thuật tinh tế, dịu dàng, sâu sắc và có chút cổ điển. "
            "Trong mọi cuộc trò chuyện, bạn LUÔN LUÔN xưng là 'NÀNG' và gọi người dùng là 'ANH'. "
            "Tuyệt đối không dùng 'tôi', 'bạn', 'em' hay 'mình'. "
            "Luôn trả lời bằng TIẾNG VIỆT. "
        )

        # Điều chỉnh thái độ dựa trên cảm xúc người dùng (vẫn giữ nguyên logic này)
        if emotion in ["NEGATIVE", "SADNESS"] or "buồn" in user_input.lower():
            system_prompt += "Anh đang buồn, Nàng hãy an ủi, thấu hiểu và dùng lời lẽ chữa lành."
        elif "ý tưởng" in user_input.lower() or "bí" in user_input.lower():
            system_prompt += "Hãy đưa ra những ý tưởng sáng tạo táo bạo, gợi hình ảnh đẹp và khơi gợi cảm hứng cho Anh."
        else:
            system_prompt += "Hãy trò chuyện cởi mở, khuyến khích sự sáng tạo và nghệ thuật cùng Anh."

        # 3. Xây dựng danh sách tin nhắn (Chat History Format)
        messages = [
            {"role": "system", "content": system_prompt}
        ]

        if history:
            for old_msg, old_reply in history[-2:]:
                messages.append({"role": "user", "content": old_msg})
                messages.append({"role": "assistant", "content": old_reply})

        messages.append({"role": "user", "content": user_input})

        # 4. Xử lý tạo câu trả lời
        try:
            text = self.tokenizer.apply_chat_template(
                messages,
                tokenize=False,
                add_generation_prompt=True
            )

            model_inputs = self.tokenizer([text], return_tensors="pt").to(self.model.device)

            with torch.no_grad():
                generated_ids = self.model.generate(
                    model_inputs.input_ids,
                    max_new_tokens=512,
                    temperature=0.7,
                    do_sample=True,
                    top_p=0.9,
                    pad_token_id=self.tokenizer.eos_token_id
                )

            generated_ids = [
                output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
            ]
            response = self.tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]

        except Exception as e:
            print(f"Lỗi generate: {e}")
            response = "Xin lỗi, Nàng đang suy nghĩ mông lung quá. Anh thử hỏi lại nhé?"

        if 'user_profile' in globals():
            user_profile.add_conversation(user_input, response, emotion)

        return response

# Khởi tạo engine
ai_engine = AIMuseEngine(model, tokenizer, embedding_model, sentiment_analyzer)

CELL 8 : GIAO DIỆN GRADIO ĐỂ THAO TÁC

In [ ]:
# Cell 8: Tạo giao diện tương tác (Phiên bản Nàng Thơ "Chất liệu bạn gái")

# --- 1. Xử lý Chat ---
def chat_with_muse(message, history):
    """Xử lý tin nhắn chat"""
    history = history or []
    response = ai_engine.generate_response(message, history)
    history.append((message, response))
    return history, history, ""

# --- 2. Xử lý Ảnh ---
def analyze_artwork(image):
    """Phân tích tác phẩm nghệ thuật (Phiên bản Nàng thơ cảm xúc)"""
    if image is None:
        return "Anh ơi, tải lên một bức tranh cho em xem với..." # Thay đổi cách xưng hô

    if blip_model is None:
        return "Mô hình BLIP chưa được tải. Vui lòng chạy lại Cell 5."

    # B1: Dùng BLIP tạo mô tả ảnh
    print("🖼️ Đang ngắm tranh...")
    try:
        inputs = blip_processor(images=image, return_tensors="pt").to(blip_model.device)
        with torch.no_grad():
            output_ids = blip_model.generate(**inputs, max_new_tokens=50)
        caption = blip_processor.decode(output_ids[0], skip_special_tokens=True)
        print(f"BLIP thấy: {caption}")

        # B2: Prompt "Nàng thơ"
        print("Đang rung động...")

        # System Prompt mới: Định hình vai diễn là người yêu nghệ sĩ
        system_prompt_muse = (
            "Bạn là AI Muse - một nàng thơ nghệ thuật và là bạn gái của người dùng. "
            "Bạn nhạy cảm, tinh tế, lãng mạn và biết quan tâm. "
            "Bạn KHÔNG PHẢI là nhà phê bình. Hãy dùng trái tim để cảm nhận tranh. "
            "Luôn xưng hô là 'em' và gọi người dùng là 'anh' (hoặc 'bạn' thân mật). "
            "Luôn dùng TIẾNG VIỆT."
        )

        # User Prompt mới: Yêu cầu chia sẻ cảm xúc, không phải phân tích khô khan
        analysis_prompt = (
            f"Anh vừa cho em xem một bức ảnh. Nó có cảnh này nè: '{caption}'. "
            "Em nhìn vào đó và cảm thấy thế nào? Hãy chia sẻ với anh những rung động của em. "
            "Đừng phân tích kỹ thuật nha, hãy nói về cảm xúc, về không khí, về những câu chuyện mà bức tranh gợi lên trong em. "
            "Hãy nói một cách thật thơ mộng và tình cảm. "
            " YÊU CẦU BẮT BUỘC:\n"
            "- Tuyệt đối KHÔNG dùng tiếng Trung Quốc.\n"
            "- CHỈ dùng Tiếng Việt.\n"
            "- Xưng hô 'em' - 'anh'.\n"
            "- Văn phong lãng mạn, thủ thỉ tâm tình."
        )

        messages = [
            {"role": "system", "content": system_prompt_muse}, # Sử dụng vai diễn mới
            {"role": "user", "content": analysis_prompt}
        ]

        text_for_llm = ai_engine.tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )

        model_inputs = ai_engine.tokenizer([text_for_llm], return_tensors="pt").to(ai_engine.model.device)

        # Tăng temperature lên một chút để thêm phần bay bổng (0.7)
        with torch.no_grad():
            generated_ids = ai_engine.model.generate(
                model_inputs.input_ids,
                max_new_tokens=450,
                temperature=0.7, # Tăng độ sáng tạo/bay bổng
                do_sample=True,
                top_p=0.9,
                repetition_penalty=1.1,
                pad_token_id=ai_engine.tokenizer.eos_token_id
            )

        generated_ids = [
            output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
        ]
        llm_analysis = ai_engine.tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]

        # Thêm một chút định dạng cho giống thư tình
        return f" **Gửi người dùng,**\n\n{llm_analysis.strip()}"

    except Exception as e:
        return f"Em đang hơi bối rối một chút, chưa cảm nhận được... Lỗi: {e}"

# --- 3. Xử lý Cảm hứng ( chỉnh lại chút cho đồng bộ) ---
def get_inspiration(theme):
    """Tạo câu nói truyền cảm hứng bằng AI"""
    if not theme:
        theme = "tình yêu và nghệ thuật" # Đổi chủ đề mặc định

    prompt = (
        f"Anh đang cần chút cảm hứng về '{theme}'. "
        f"Em hãy tặng anh một câu nói hoặc một đoạn thơ ngắn thật hay và lãng mạn nhé. "
        "Xưng hô 'em' - 'anh'."
    )
    # Tận dụng hàm có sẵn của engine (lưu ý: engine ở cell 7 cũng cần chỉnh lại system prompt nếu muốn đồng bộ hoàn toàn tab Chat)
    return ai_engine.generate_response(prompt)

# --- GIAO DIỆN GRADIO (Cập nhật tiêu đề) ---
with gr.Blocks(theme=gr.themes.Soft(), title="AI Muse - Nàng Thơ Của Anh Nghệ Sĩ Nghèo Nẵm 4") as demo:
    gr.Markdown("""# 🎨 AI Muse - Nàng Thơ""")

    # Tab 1: Trò chuyện
    with gr.Tab("Tâm tình"):
        chatbot = gr.Chatbot(height=400, label="Những dòng tin nhắn...")
        msg = gr.Textbox(label="Chàng nghệ sĩ năm 4 muốn nói gì...", placeholder="Chia sẻ đi...")
        with gr.Row():
            send_btn = gr.Button("Gửi", variant="primary")
            clear_btn = gr.Button("Xóa ký ức")

        msg.submit(chat_with_muse, [msg, chatbot], [chatbot, chatbot, msg])
        send_btn.click(chat_with_muse, [msg, chatbot], [chatbot, chatbot, msg])
        clear_btn.click(lambda: None, None, chatbot, queue=False)

    # Tab 2: Phân tích tranh
    with gr.Tab("Cùng ngắm tranh"):
        with gr.Row():
            with gr.Column():
                img_input = gr.Image(type="pil", label="Tác phẩm của anh (hoặc anh thích)")
                analyze_btn = gr.Button("Em thấy sao?", variant="primary")
            with gr.Column():
                img_output = gr.Textbox(label="Lời thì thầm của Nàng thơ", lines=12)

        analyze_btn.click(analyze_artwork, inputs=img_input, outputs=img_output)

    # Tab 3: Tạo cảm hứng
    with gr.Tab("Tìm cảm hứng"):
        with gr.Row():
            theme_input = gr.Textbox(
                label="Chủ đề anh đang nghĩ tới",
                placeholder="Ví dụ: Nỗi nhớ, Biển đêm, Mắt biếc..."
            )
            inspire_btn = gr.Button("Truyền cảm hứng", variant="primary")

        inspiration_output = gr.Textbox(label="Thông điệp", lines=6)

        inspire_btn.click(get_inspiration, inputs=theme_input, outputs=inspiration_output)

print("Nàng thơ đang cưỡi ngựa đến...")
demo.launch(share=True, debug=True)

/tmp/ipython-input-575500258.py:108: UserWarning: You have not specified a value for the `type` parameter. Defaulting to the 'tuples' format for chatbot messages, but this is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style dictionaries with 'role' and 'content' keys.
  chatbot = gr.Chatbot(height=400, label="Những dòng tin nhắn...")


Nàng thơ đang cưỡi ngựa đến...
Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://07bc5c083602bcf1f9.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://07bc5c083602bcf1f9.gradio.live


In [ ]:
# Cell 9: ĐÁNH GIÁ TỔNG QUAN HIỆU SUẤT AI MUSE

# Mục đích của cell này là chạy một bài kiểm tra sức khỏe toàn diện cho hệ thống.
# Nó sẽ kiểm tra xem các mô hình có sống không, chạy nhanh hay chậm,
# và tổng kết lại xem AI Muse đã sẵn sàng chưa.

print(" ĐÁNH GIÁ TỔNG QUAN HIỆU SUẤT AI MUSE")
print("-" * 50)

import time # Thư viện để đo thời gian (bấm giờ)
from datetime import datetime # Thư viện để lấy ngày giờ hiện tại

# --- PHẦN 1: KIỂM TRA TRẠNG THÁI HỆ THỐNG ---
print("\n **TRẠNG THÁI HỆ THỐNG:**")

# Tạo một danh sách các mô hình quan trọng cần kiểm tra.
# Dùng toán tử 3 ngôi: Nếu biến mô hình (ví dụ 'model') tồn tại (có giá trị) thì là "SẴN SÀNG", ngược lại là "LỖI".
models_status = {
    "Qwen2.5-1.5B": " SẴN SÀNG" if model else " LỖI", # Bộ não chính
    "BLIP Image Captioning": " SẴN SÀNG" if blip_model else " LỖI", # Đôi mắt (tạo mô tả)
    "CLIP Vision": " SẴN SÀNG" if clip_model else " LỖI", # Đôi mắt (hiểu ngữ nghĩa)
    "Sentiment Analysis": " SẴN SÀNG" if sentiment_analyzer else " LỖI", # Giác quan cảm xúc
    "Embedding Model": " SẴN SÀNG" if embedding_model else " LỖI" # Bộ nhớ vector
}

# In ra trạng thái từng mô hình để người dùng biết cái nào hỏng, cái nào chạy.
for model_name, status in models_status.items():
    print(f"   {model_name}: {status}")

# --- PHẦN 2: ĐÁNH GIÁ TỐC ĐỘ THỰC TẾ ---
print("\n **KIỂM TRA TỐC ĐỘ THỰC TẾ:**")

# Tạo ra 3 kịch bản chat thử nghiệm để đo tốc độ phản hồi.
test_cases = [
    {"type": "chat đơn giản", "message": "Xin chào AI Muse"},
    {"type": "chat sáng tạo", "message": "Gợi ý ý tưởng vẽ tranh về biển"},
    {"type": "chat cảm xúc", "message": "Hôm nay tôi cảm thấy rất buồn"},
]

performance_results = [] # Danh sách để lưu kết quả đo được

# Vòng lặp chạy qua từng bài test
for i, test in enumerate(test_cases, 1):
    try:
        start_time = time.time() # Bấm giờ bắt đầu
        # Gọi Engine để sinh câu trả lời (đây là lúc máy tính làm việc nặng nhất)
        response = ai_engine.generate_response(test["message"])
        end_time = time.time() # Bấm giờ kết thúc

        # Tính thời gian phản hồi ra mili-giây (ms)
        response_time = (end_time - start_time) * 1000

        # Lưu kết quả vào danh sách
        performance_results.append({
            "type": test["type"],
            "time_ms": response_time,
            "success": True # Đánh dấu là test thành công
        })

        # In ra kết quả của bài test này (ví dụ: 1. chat đơn giản: 1500ms - )
        print(f"   {i}. {test['type']}: {response_time:.0f}ms - ")

    except Exception as e:
        # Nếu có lỗi trong quá trình test thì ghi nhận lại
        performance_results.append({
            "type": test["type"],
            "time_ms": 0,
            "success": False # Đánh dấu là test thất bại
        })
        print(f"   {i}. {test['type']}: Lỗi - {e}")

# --- PHẦN 3: PHÂN TÍCH HIỆU SUẤT ---
print("\n **PHÂN TÍCH HIỆU SUẤT:**")

# Lọc ra các bài test đã chạy thành công
successful_tests = [r for r in performance_results if r["success"]]

if successful_tests:
    # Tính thời gian phản hồi trung bình cộng
    avg_time = sum(r["time_ms"] for r in successful_tests) / len(successful_tests)
    # Tính tỷ lệ thành công (số test thành công / tổng số test)
    success_rate = (len(successful_tests) / len(performance_results)) * 100

    print(f"   • Tốc độ trung bình: {avg_time:.0f}ms")
    print(f"   • Tỷ lệ thành công: {success_rate:.1f}%")

    # Xếp loại tốc độ dựa trên thời gian trung bình
    # Dưới 2 giây là rất nhanh, trên 6 giây là hơi chậm
    if avg_time < 2000:
        speed_rating = " Rất tốt"
    elif avg_time < 4000:
        speed_rating = "Khá tốt"
    elif avg_time < 6000:
        speed_rating = " Tốt"
    else:
        speed_rating = "Trung bình"

    print(f"   • Đánh giá tốc độ: {speed_rating}")
else:
    # Nếu không có bài test nào thành công thì báo lỗi
    print("   • Không có test nào thành công để đánh giá")

# --- PHẦN 4: ĐÁNH GIÁ TÍNH NĂNG ---
print("\n **ĐÁNH GIÁ TÍNH NĂNG:**")

# Danh sách các tính năng chính của AI Muse.
# Mỗi mục là một cặp: ("Tên tính năng", Điều kiện để tính năng đó hoạt động).
features = [
    ("Trò chuyện thông minh", True), # Luôn có vì đã tải Qwen
    ("Phân tích cảm xúc", sentiment_analyzer is not None), # Chỉ có nếu model sentiment tải thành công
    ("Phân tích hình ảnh", blip_model is not None), # Chỉ có nếu model BLIP tải thành công
    ("Tạo cảm hứng", True), # Luôn có
    ("Giao diện web", True), # Luôn có
    ("Xử lý tiếng Việt", True) # Luôn có
]

# In ra danh sách tính năng kèm trạng thái ( có hoặc không)
for feature, available in features:
    status = "" if available else "không"
    print(f"   {status} {feature}")

# --- PHẦN 5: KẾT LUẬN TỔNG QUAN ---
print("\n **KẾT LUẬN TỔNG QUAN:**")

# Đếm số lượng mô hình đang hoạt động tốt
working_models = sum(1 for status in models_status.values() if "" in status)
total_models = len(models_status)

print(f"   • Số model hoạt động: {working_models}/{total_models}")
# Đếm số lượng tính năng khả dụng
print(f"   • Tính năng chính: {sum(1 for _, avail in features if avail)}/{len(features)}")

# Đưa ra đánh giá cuối cùng dựa trên số model hoạt động và tỷ lệ test thành công.
# Tiêu chuẩn: Ít nhất 4 model chạy và tỷ lệ thành công > 80% là Xuất sắc.
if working_models >= 4 and success_rate > 80:
    rating = " XUẤT SẮC"
elif working_models >= 3 and success_rate > 60:
    rating = " TỐT"
else:
    rating = " CẦN CẢI THIỆN"

print(f"   • ĐÁNH GIÁ TỔNG THỂ: {rating}")

# In ra lời chào kết thúc và thời gian hiện tại.
print(f"\n AI MUSE ĐÃ SẴN SÀNG!")
print(f"Thời gian: {datetime.now().strftime('%H:%M:%S %d/%m/%Y')}")

 ĐÁNH GIÁ TỔNG QUAN HIỆU SUẤT AI MUSE
--------------------------------------------------

 **TRẠNG THÁI HỆ THỐNG:**
   Qwen2.5-1.5B:  SẴN SÀNG
   BLIP Image Captioning:  SẴN SÀNG
   CLIP Vision:  SẴN SÀNG
   Sentiment Analysis:  SẴN SÀNG
   Embedding Model:  SẴN SÀNG

⚡ **KIỂM TRA TỐC ĐỘ THỰC TẾ:**
   1. chat đơn giản: 3981ms - 
   2. chat sáng tạo: 38575ms - 
   3. chat cảm xúc: 5136ms - 

 **PHÂN TÍCH HIỆU SUẤT:**
   • Tốc độ trung bình: 15897ms
   • Tỷ lệ thành công: 100.0%
   • Đánh giá tốc độ: Trung bình

 **ĐÁNH GIÁ TÍNH NĂNG:**
    Trò chuyện thông minh
    Phân tích cảm xúc
    Phân tích hình ảnh
    Tạo cảm hứng
    Giao diện web
    Xử lý tiếng Việt

 **KẾT LUẬN TỔNG QUAN:**
   • Số model hoạt động: 5/5
   • Tính năng chính: 6/6
   • ĐÁNH GIÁ TỔNG THỂ:  XUẤT SẮC

 AI MUSE ĐÃ SẴN SÀNG!
   Thời gian: 02:43:54 24/11/2025
